### THE MASTER DATA

5 dimension tables are created namely:

- dim_hub_hierarchy
- dim_node
- dim_products
- dim_scanner
- dim_supplier


In [1]:
import yaml
import random
import numpy as np 
import pandas as pd
from datetime import date, timedelta
from faker import Faker
from schema import(
    ProductSchema, SupplierSchema, NodeSchema,
    ScannerSchema, HubHierarchySchema
)
from coordinates import get_coordinates, zone_coordinates

fake = Faker("en_IN")

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

random.seed(cfg['simulation']["random_seed"])
np.random.seed(cfg["simulation"]["random_seed"])

SIM_START = date(2025,12,1)
SIM_END = date(2026,6,30)

print("Ready. ")

Ready. 


#### **Generate dim_supplier**

In [2]:
SUPPLIER_CATALOG = [
    # (name, city, edi_format, categories)
    # --------DAIRY--------
    ("Amul Cooperative North", "Delhi", "API", "dairy"),
    ("Amul Cooperative West", "Mumbai", "API", "dairy,frozen"),
    ("Amul Cooperative South", "Bangalore", "SFTP_CSV", "dairy"),
    ("Mother Dairy", "Delhi", "API", "dairy"),
    ("Nandini Dairy", "Bangalore", "SFTP_CSV", "dairy"),
    ("Fresh Farms Local", "Mohali", "email_manual", "dairy"),


    # --------Beverages--------
    ("Bisleri International", "Mumbai", "SFTP_CSV", "beverages"),
    ("Coca-Cola India", "Pune", "API", "beverages"),
    ("PepsiCo India", "Hyderabad", "API", "beverages"),
    ("Sunrise Distrbiutors", "Indore", "email_manual", "beverages,snacks"), 

    # --------STAPLES--------
    ("Fortune Foods", "Ahmedabad", "SFTP_CSV", "staples"),
    ("ITC Agri Business", "Hyderabad", "SFTP_CSV", "staples,snacks"),
    ("Ramesh Traders", "Surat", "email_manual", "staples,snacks"),

    # --------SNACKS--------
    ("Haldiram Snacks", "Nagpur", "SFTP_CSV", "snacks"),

    # --------Personal CARE AND HOUSEHOLD --------
    ("Hindustan Unilever", "Mumbai", "SFTP_CSV", "personal_care,household"),
    ("Colgate Palmolive", "Mumbai", "SFTP_XML", "personal_care,household"),
    ("Himalaya Wellness", "Bangalore", "SFTP_XML", "personal_care"),

    # -------FROZEN--------
    ("McCain Foods", "Pune", "SFTP_XML", "frozen"),
    ("Godrej Tyson Foods", "Mumbai", "SFTP_XML", "frozen"),

]

# mother hub id is needed for serves_mother_hub_ids
# assigned geographically - suppliers serve hubs in their region
SUPPLIER_HUB_MAP = {

    "Delhi": "MH_DEL_01",
    "Mumbai": "MH_MUM_01",
    "Bangalore": "MH_BLR_01",
    "Hyderabad": "MH_HYD_01",
    "Pune": "MH_MUM_01,MH_PUN_01",
    "Ahmedabad": "MH_AMD_01",
    "Nagpur": "MH_MUM_01,MH_HYD_01",
    "Surat": "MH_AMD_01",
    "Mohali": "MH_DEL_01",
    "Indore": "MH_AMD_01,MH_HYD_01",

}
CATEGORY_SUPPLIER_MAP = {
    "dairy": ["SUP_001", "SUP_002", "SUP_003", "SUP_004", "SUP_005", "SUP_006"],
    "beverages": ["SUP_007", "SUP_008", "SUP_009", "SUP_010"],
    "staples": ["SUP_011","SUP_012","SUP_013"],
    "snacks": ["SUP_010","SUP_013","SUP_014"],
    "personal_care": ["SUP_015","SUP_016","SUP_017"],
    "household": ["SUP_015","SUP_016"],
    "frozen": ["SUP_002","SUP_018","SUP_019"],
}

edi_cfg = cfg["suppliers"]["edi_formats"]

def generate_suppliers():
    suppliers = []
    for i, (name, city, edi, categories) in enumerate(SUPPLIER_CATALOG, 1):
        score_range = edi_cfg[edi]['quality_score_range']
        score = round(random.uniform(score_range[0], score_range[1]), 3)

        ## onboarded 1-5 years before simulation start
        days_back = random.randint(365, 1825)
        onboarded = SIM_START - timedelta(days = days_back)

        raw = {
            "supplier_id": f"SUP_{i:03d}",
            "supplier_name": name,
            "city": city,
            "edi_format": edi,
            "data_quality_score": score,
            "onboarded_date": onboarded,
            "categories_supplied": categories,
            "serves_mother_hub_ids": SUPPLIER_HUB_MAP.get(city, "MH_MUM_01")
        }

        validated = SupplierSchema(**raw)
        suppliers.append(validated.model_dump())

    return pd.DataFrame(suppliers)

df_supplier = generate_suppliers()
print(f"Suppliers: {len(df_supplier)}")
df_supplier.head()

Suppliers: 19


,supplier_id,supplier_name,city,edi_format,data_quality_score,onboarded_date,categories_supplied,serves_mother_hub_ids
0,SUP_001,Amul Cooperative North,Delhi,API,0.940,2024-10-11,dairy,MH_DEL_01
1,SUP_002,Amul Cooperative West,Mumbai,API,0.954,2023-07-19,"dairy,frozen",MH_MUM_01
2,SUP_003,Amul Cooperative South,Bangalore,SFTP_CSV,0.695,2024-05-06,dairy,MH_BLR_01
3,SUP_004,Mother Dairy,Delhi,API,0.945,2021-11-11,dairy,MH_DEL_01
4,SUP_005,Nandini Dairy,Bangalore,SFTP_CSV,0.667,2022-07-21,dairy,MH_BLR_01


### **Generate dim_product**

In [3]:
PRODUCT_CATALOG = {
    "dairy": {
        "items": [
            # (name, U0M, min_price, max_price, shelf_life, weight_g, pack_ml_g, [list of brands])
            ("Full Cream Milk 500ml", "units", 24, 28, 7, 520, 500,
             ["Amul", "Mother Dairy", "Nandini"]),
            ("Full Cream Milk 1L", "units", 44, 52, 7, 1050, 1000,
             ["Amul", "Mother Dairy", "Nandini"]),
            ("Toned Milk 500ml", "units", 20, 24, 7, 520, 500,
             ["Amul", "Mother Dairy", "Verka"]),
            ("Paneer 200g", "units", 75, 95, 10, 210, None,
             ["Amul", "Mother Dairy", "Nandini"]),
            ("Butter 100g", "units", 52, 58, 90, 110, None,
             ["Amul", "Mother Dairy", "Parag"]),
            ("Curd 400g", "units", 38, 48, 5, 420, None,
             ["Amul", "Mother Dairy", "Verka"]),
            ("Ice cream tub 500g","units", 350, 440, 180, 500, 500,
             ["Amul", "Kwality Walls", "Havmor"]),
        ]
    },

    "beverages": {
        "items": [
            ("Packaged Water 1L", "units", 15, 20, 365, 1050, 1000,
             ["Bisleri", "Kinley", "Aquafina"]),
            ("Packaged Water 500ml", "units", 10, 14, 365, 530, 500,
             ["Bisleri", "Kinley", "Aquafina"]),
            ("Cola 500ml", "units", 30, 40, 180, 540, 500,
             ["Pepsi", "Coca-Cola", "Thums Up"]),
            ("Fruit Juice 200ml", "units", 18, 28, 120, 220, 200,
             ["Paper Boat", "Real", "Tropicana"]),
            ("Energy Drink 250ml", "units", 85, 110, 180, 275, 250,
             ["Red Bull", "Monster"]),
        ] # to do: figure out pepsi and coca cola as suppliers for bisleri and kinley;tropicana and thums up; 
    },

    "staples": {
        "items": [
            ("Basmati Rice 1kg", "kg", 85, 110, 365, 1020, None,
             ["India Gate", "Fortune", "Daawat"]),
            ("Toor Dal 500g", "units", 58, 78, 365, 520, None,
             ["Tata Sampann", "Fortune", "Aashirvaad"]),
            ("Atta 1kg", "kg", 48, 62, 180, 1020, None,
            ["Aashirvaad", "Pillsbury", "Fortune"] ),
            ("Sunflower Oil 1L", "units", 120, 155, 365, 1050, 1000,
             ["Fortune", "Saffola", "Sundrop"]),
            ("Sugar 1kg", "kg", 42, 52, 730, 1020, None,
             ["Uttam", "Dhampur", "Fortune"]),
        ]
    },

    "snacks": {
        "items": [
            ("Namkeen Mix 200g", "units", 38, 58, 180, 205, None,
             ["Haldiram", "Bikaji", "Balaji"]),
            ("Potato Chips 100g", "units", 18, 28, 90, 105, None,
             ["Lays", "Bingo", "Too Yum"]),
            ("Roasted Peanuts 150g", "units", 28, 42, 180, 158, None,
             ["Haldiram", "Bikaji", "Jabsons"]),
            ("Biscuits 200g", "units", 22, 35, 180, 210, None,
             ["Parle G", "Brittania", "Sunfeast"]),
            ("Instant Noodles 70g", "units", 12, 20, 180, 75, None, 
             ["Maggi", "Yipee", "Wai Wai"])
        ] # Lays would come under pepsico supplier, 
        # choosing a random supplier for multiple snacks now because suppliers are increasing as i create diverse products;
    },

    "household": {
        "items": [
            ("Dishwash Liquid 500ml", "units", 78, 120, 730, 530, 500,
             ["Vim", "Prill", "Exo"]),
            ("Floor Cleaner 1L", "units", 88, 140, 730, 1060, 1000,
             ["Lizol", "Colin", "Domex"]),
            ("Detergent Powder 1kg", "kg", 88, 130, 730, 1020, None,
             ["Surf Excel", "Ariel", "Tide"]),
            ("Toilet Cleaner 500ml", "units", 68, 98, 730, 540, 500, 
             ["Harpic", "Domex", "Sanifresh"]),
        ]
    },

    "personal_care": {
        "items": [
            ("Shampoo 200ml", "units", 120, 220, 730, 215, 200,
             ["Dove", "Head and Shoulders", "Pantene"]),
            ("Hand Wash 250ml", "units", 65, 110, 730, 265, 250,
             ["Dettol", "Lifebuoy", "Savlon"]),
            ("Toothpaste 100g", "units", 55, 95, 730, 108, None,
             ["Colgate", "Pepsodent", "Sensodyne"]),
            ("Face Wash 100ml", "units", 85, 160, 730, 108, 100,
             ["Himalaya", "Cetaphil", "Nivea"]),
            ("Moisturizer 50ml", "units", 120, 200, 730, 54, 50,
             ["Nivea", "Vaseline", "Dove"]),
        ]
    },

    "frozen": {
        "items": [
            ("Frozen Peas 500g", "units", 58, 88, 180, 520, None,
             ["McCain", "Mother Dairy", "ITC"]),
            ("Chicken Nuggets 400g", "units", 175, 245, 90, 420, None,
             ["Venky's", "Godrej Yummiez", "McCain"]),
            ("Veg Burger Patty 300g", "units", 95, 135, 90, 315, None,
             ["McCain", "ITC", "Amul"]),
            ("French Fries 400g", "units", 75, 120, 200,420 ,None,
             ["McCain", "Haldiram", "ITC"])
        ]
    },
}

# map category to supplier_id
CATEGORY_SUPPLIER_MAP = {
    "dairy": ["SUP_001", "SUP_002", "SUP_003", "SUP_004", "SUP_005", "SUP_006"],
    "beverages": ["SUP_007", "SUP_008", "SUP_009", "SUP_010"],
    "staples": ["SUP_011","SUP_012","SUP_013"],
    "snacks": ["SUP_010","SUP_013","SUP_014"],
    "personal_care": ["SUP_015","SUP_016","SUP_017"],
    "household": ["SUP_015","SUP_016"],
    "frozen": ["SUP_002","SUP_018","SUP_019"],
} # this is mayhem now, it needs to be corrected here too

prod_cfg = cfg["product_categories"]

def generate_products():
    products = []
    prod_num = 1
    
    for category, data in PRODUCT_CATALOG.items():
        cat_cfg = prod_cfg[category]
        items = data["items"]
        suppliers = CATEGORY_SUPPLIER_MAP[category]

        for item in items:
            name, uom, min_p, max_p, shelf, weight, pack, item_brands = item
            for brand in item_brands[:3]: #top 3 brands per item
                if prod_num > cfg["products"]["total"]:
                    break

                price = round(random.uniform(min_p, max_p), 2)
                reorder = random.randint(20, 80)

                raw = {
                    "product_id": f"PROD_{prod_num:03d}",
                    "name": f"{brand} {name}",
                    "category": category,
                    "brand": brand,
                    "unit_of_measure": uom,
                    "base_price": price,
                    "pack_size_ml_or_g": pack,
                    "weight_grams": weight,
                    "shelf_life_days": shelf,
                    "storage_zone": cat_cfg["storage_zone"],
                    "reorder_point_units": reorder,
                    "supplier_id": random.choice(suppliers),
                }

                validated = ProductSchema(**raw)
                products.append(validated.model_dump())
                prod_num +=1

            if prod_num > cfg["products"]["total"]:
                break
        if prod_num > cfg["products"]["total"]:
            break

    return pd.DataFrame(products)

df_product = generate_products()
print(f"Products: {len(df_product)}")
df_product.head(10)   

Products: 100


,product_id,name,category,brand,unit_of_measure,base_price,pack_size_ml_or_g,weight_grams,shelf_life_days,storage_zone,reorder_point_units,supplier_id
0,PROD_001,Amul Full Cream Milk 500ml,dairy,Amul,units,26.92,500.0,520,7,cold,54,SUP_001
1,PROD_002,Mother Dairy Full Cream Milk 500ml,dairy,Mother Dairy,units,27.89,500.0,520,7,cold,44,SUP_001
2,PROD_003,Nandini Full Cream Milk 500ml,dairy,Nandini,units,26.21,500.0,520,7,cold,73,SUP_006
3,PROD_004,Amul Full Cream Milk 1L,dairy,Amul,units,48.95,1000.0,1050,7,cold,75,SUP_003
4,PROD_005,Mother Dairy Full Cream Milk 1L,dairy,Mother Dairy,units,48.62,1000.0,1050,7,cold,65,SUP_001
5,PROD_006,Nandini Full Cream Milk 1L,dairy,Nandini,units,44.37,1000.0,1050,7,cold,34,SUP_003
6,PROD_007,Amul Toned Milk 500ml,dairy,Amul,units,23.94,500.0,520,7,cold,74,SUP_002
7,PROD_008,Mother Dairy Toned Milk 500ml,dairy,Mother Dairy,units,23.47,500.0,520,7,cold,44,SUP_003
8,PROD_009,Verka Toned Milk 500ml,dairy,Verka,units,21.81,500.0,520,7,cold,73,SUP_003
9,PROD_010,Amul Paneer 200g,dairy,Amul,units,78.25,NaN,210,10,cold,42,SUP_002


### **Generate dim_node (Mother_hubs)**

In [4]:
def generate_mother_hubs():
    nodes = []
    mh_cfg = cfg["mother_hubs"]

    for city, loc in mh_cfg["locations"].items():
        city_code = loc["city_code"]
        zone = loc["zone"]
        zone_code = loc["zone_code"]
        node_id = f"MH_{city_code}_01"

        total = random.randint(
            mh_cfg["capacity_total_units"][0],
            mh_cfg["capacity_total_units"][1],
        )

        cold_frac = random.uniform(
        mh_cfg["cold_fraction"][0],
        mh_cfg["cold_fraction"][1]
        )

        bev_frac = random.uniform(
            mh_cfg["beverage_fraction"][0],
            mh_cfg["beverage_fraction"][1]
        )

        if cold_frac + bev_frac > 0.85:
            bev_frac = 0.85-cold_frac

        cold = int(total* cold_frac)
        bev = int(total * bev_frac)
        dry = total - cold - bev

        lat, lon = get_coordinates(zone, city, "mother_hub", jitter = False)

        # opened 3-7 years before simulation start
        op_since = SIM_START - timedelta(days= random.randint(1095, 2555))

        raw = {
            "node_id": node_id,
            "node_type": "mother_hub",
            "node_name": f"{city} Mother Hub",
            "city": city,
            "zone": zone,
            "zone_type": "warehouse",
            "tier": "metropolitan",
            "parent_node_id": None,
            "capacity_units": total,
            "capacity_cold_units": cold,
            "capacity_beverage_units": bev,
            "capacity_dry_units": dry,
            "lat": lat,
            "lon": lon,
            "timezone": "Asia/Kolkata",
            "operational_since": op_since,
        }

        validated = NodeSchema(**raw)
        nodes.append(validated.model_dump())

    return nodes

mother_hub_rows = generate_mother_hubs()
print(f"Mother hubs: {len(mother_hub_rows)}")

Mother hubs: 8


### **Generate Regional Hubs**

In [5]:
# define which zones each regional hub serves
# used later to assign darkstores to regional hubs

RH_SERVICE_ZONES = {
    "MH_MUM_01": {
        "RH_MUM_WEST_01": ["Andheri", "Bandra", "Malad", "Borivali"],
        "RH_MUM_EAST_01": ["Thane", "Kurla"],
    },
    "MH_DEL_01": {
        "RH_DEL_NORTH_01": ["Rohini", "Janakpuri", "Dwarka"],
        "RH_DEL_SOUTH_01": ["Saket", "Lajpat Nagar"],
    },
    "MH_BLR_01": {
        "RH_BLR_EAST_01": ["Whitefield", "Koramangala"],
        "RH_BLR_WEST_01": ["Indiranagar", "HSR Layout"],
    },
    "MH_HYD_01": {
        "RH_HYD_WEST_01": ["Kondapur", "Madhapur", "Gachibowli"],
        "RH_HYD_EAST_01": ["Banjara Hills"],

    },
    "MH_CHN_01": {
        "RH_CHN_01": ["Anna Nagar", "T Nagar", "Velachery", "Adyar"],
    },
    "MH_KOL_01": {
        "RH_KOL_01": ["Salt Lake", "Park Street", "New Town", "Behala"],
    },

    ## tier1 cities to have one regional hub per city
    "MH_PUN_01": {"RH_PUN_01": ["Kothrud", "Wakad", "Baner", "Hadapsar"]},
    "MH_AMD_01": {"RH_AMD_01": ["Navrangpura", "Satellite", "Bopal", "Prahlad Nagar"]},
   
}

def get_zone_tier(zone_or_city, cfg):
    """Look up tier for a zone from config."""
    for city, city_data in cfg["cities"]["metropolitan"].items():
        if zone_or_city in city_data["zone_type_tags"]:
            return "metropolitan"
        if zone_or_city == city:
            return "metropolitan"
    for city, city_data in cfg["cities"]["tier_1"].items():
        if zone_or_city in city_data["zone_type_tags"]:
            return "tier_1"
    return "tier_2"

def generate_regional_hubs():
    nodes = []
    rh_cfg = cfg["regional_hubs"]

    for mother_hub_id, rh_dict in RH_SERVICE_ZONES.items():
        #infer city from mother hub id
        city_code = mother_hub_id.split("_")[1] # so MH_MUM_01 becomes MUM
        # map city code back to city name
        city_map = {
            loc["city_code"]: city
            for city, loc in cfg["mother_hubs"]["locations"].items()
            
        }

        # tier1 cities need adding
        tier1_code_map = {
            "PUN": "Pune", "AMD": "Ahmedabad", "JAI": "Jaipur",
            "LKO": "Lucknow", "CHD": "Chandigarh", "SRT": "Surat"
        }

        city_map.update(tier1_code_map)
        city = city_map.get(city_code, city_code)

        tier = "tier_1" if mother_hub_id in [
            "MH_PUN_01", "MH_AMD_01"
        ] else "metropolitan"

        cap_range = rh_cfg["capacity_total_units"][tier]
        cold_range = rh_cfg["cold_fraction"][tier]
        bev_range = rh_cfg["beverage_fraction"][tier]

        for rh_id, zones in rh_dict.items():
            total = random.randint(cap_range[0], cap_range[1])
            cold_frac = random.uniform(cold_range[0], cold_range[1])
            bev_frac = random.uniform(bev_range[0], bev_range[1])
            if cold_frac + bev_frac > 0.85:
                bev_frac = 0.85 - cold_frac

            cold = int(total* cold_frac)
            bev = int(total * bev_frac)
            dry = total - cold - bev

            # use first zone as coordinate anchor
            anchor_zone = zones[0]
            lat, lon = get_coordinates(anchor_zone, city, "regional_hub", jitter = False)

            op_since = SIM_START - timedelta(days = random.randint(730, 2190))

            raw = {
                "node_id": rh_id,
                "node_type": "regional_hub",
                "node_name": f"{rh_id.replace('_',' ')} Regional Hub",
                "city": city, 
                "zone": anchor_zone,
                "zone_type": cfg["cities"]["metropolitan"].get(city, {}).get("zone_type_tags", {}).get(
                    anchor_zone,{}).get("zone_type","commercial"),
                "tier": tier,
                "parent_node_id": mother_hub_id,
                "capacity_units": total,
                "capacity_cold_units": cold,
                "capacity_beverage_units": bev,
                "capacity_dry_units": dry,
                "lat": lat,
                "lon": lon,
                "timezone": "Asia/Kolkata",
                "operational_since": op_since,
            }

            validated = NodeSchema(**raw)
            nodes.append(validated.model_dump())

    return nodes
regional_hubs_rows = generate_regional_hubs()
print(f"Regional hubs: {len(regional_hubs_rows)}")



Regional hubs: 12


### ***Generate Darkstores***

In [6]:
# build a flat lookup: zone -> (regional_hub_id, mother_hub_id, city, tier)
ZONE_TO_RH = {}
for mh_id, rh_dict in RH_SERVICE_ZONES.items():
    for rh_id, zones in rh_dict.items():
        for zone in zones:
            ZONE_TO_RH[zone] = rh_id

# the following dictionary exists because some darkstores were not generated and 
# i have made the closest RH their regional hub 
TIER2_FALLBACK_RH = {
    "Mohali": "RH_DEL_NORTH_01",
    "Indore": "RH_AMD_01",
    "Nagpur": "RH_PUN_01",
    "Coimbatore": "RH_CHN_01",
    "Bhopal": "RH_AMD_01",
    "Patna": "RH_KOL_01",
    "Agra": "RH_DEL_SOUTH_01"

}

def generate_darkstores():
    nodes = []
    ds_cfg = cfg["darkstores"]
    cities = cfg["cities"]
    total = cfg["darkstores"]["total"]
    dist = cfg["city_distribution"]
    n_metro =int(total * dist["metropolitan"])
    n_tier1 =int(total * dist["tier_1"])
    n_tier2 = total - n_metro - n_tier1

    sim_days = (SIM_END - SIM_START).days
    n_mid_sim = max(1, int(total *0.05)) # 5% open mid-simulation

    # track sequence per zone for node_id
    zone_counter = {}
    ds_num = 0

    def make_darkstore(zone_name, city_name, zone_cfg, tier, cap_range, cold_range, 
                       bev_range, is_mid_sim= False):
        nonlocal ds_num
        ds_num += 1

        code = zone_cfg["code"]
        zone_type = zone_cfg["zone_type"]

        seq = zone_counter.get(code, 0) + 1
        zone_counter[code] = seq
        node_id = f"DS_{code}_{seq:02d}"

        total_cap = random.randint(cap_range[0], cap_range[1])
        cold_frac = random.uniform(cold_range[0], cold_range[1])
        bev_frac = random.uniform(bev_range[0], bev_range[1])
        if cold_frac + bev_frac > 0.85:
            bev_frac = 0.85 - cold_frac

        cold = int(total_cap * cold_frac)
        bev = int(total_cap *bev_frac)
        dry = total_cap - cold - bev

        lat, lon = get_coordinates(zone_name, city_name, "darkstore", jitter = True)

        if is_mid_sim:
            op_since = SIM_START + timedelta(days= random.randint(30, sim_days))
        else:
            op_since = SIM_START - timedelta(days= random.randint(30,1825))

        parent_rh = ZONE_TO_RH.get(zone_name)
        if parent_rh is None:
            parent_rh = TIER2_FALLBACK_RH.get(city_name, "RH_DEL_NORTH_01")
        if parent_rh is None:
            # tier2 city, find or create a regional hub
            parent_rh = f"RH_{city_name[:3].upper()}_01"

        raw = {
            "node_id": node_id,
            "node_type": "darkstore",
            "node_name": zone_name,
            "city": city_name,
            "zone": zone_name,
            "zone_type": zone_type,
            "tier": tier,
            "parent_node_id": parent_rh,
            "capacity_units": total_cap,
            "capacity_cold_units": cold,
            "capacity_beverage_units": bev,
            "capacity_dry_units": dry,
            "lat": lat,
            "lon": lon,
            "timezone": "Asia/Kolkata",
            "operational_since": op_since,

        }

        validated = NodeSchema(**raw)
        return validated.model_dump()
    
    # metropolitan darkstores
    metro_zones = [
        (zone_name, city_name, zone_cfg, "metropolitan")
        for city_name, city_data, in cities["metropolitan"].items()
        for zone_name, zone_cfg in city_data["zone_type_tags"].items()

    ]

    for i in range(n_metro):
        zone_name, city_name, zone_cfg, tier = metro_zones[i % len(metro_zones)]
        is_mid = (i < n_mid_sim)
        cap = ds_cfg["capacity_total_units"]["metropolitan"]
        cold = ds_cfg["cold_fraction"]["metropolitan"]
        bev = ds_cfg["beverage_fraction"]["metropolitan"]
        nodes.append(make_darkstore(zone_name, city_name, zone_cfg,
                                    tier, cap, cold, bev, is_mid))
        
    # tier1 darkstores
    tier1_zones = [
        (zone_name, city_name, zone_cfg, "tier_1")
        for city_name, city_data, in cities["tier_1"].items()
        for zone_name, zone_cfg in city_data["zone_type_tags"].items()
    ]
    for i in range(n_tier1):
        zone_name, city_name, zone_cfg, tier = tier1_zones[i%len(tier1_zones)]
        is_mid = (i < n_mid_sim)
        cap = ds_cfg["capacity_total_units"]["tier_1"]
        cold = ds_cfg["cold_fraction"]["tier_1"]
        bev = ds_cfg["beverage_fraction"]["tier_1"]
        nodes.append(make_darkstore(zone_name, city_name, zone_cfg,
                                    tier, cap, cold, bev, is_mid))

    #tier2 darkstores
    tier2_cities = list(cities["tier_2"].items())
    for i in range(n_tier2):
        city_name, city_cfg = tier2_cities[i%len(tier2_cities)]
        zone_cfg = {"code": city_cfg["code"], "zone_type": city_cfg["zone_type"]}
        cap = ds_cfg["capacity_total_units"]["tier_2"]
        cold = ds_cfg["cold_fraction"]["tier_2"]
        bev = ds_cfg["beverage_fraction"]["tier_2"]
        nodes.append(make_darkstore(city_name, city_name, zone_cfg, "tier_2", cap, cold, bev))

    return nodes

darkstore_rows = generate_darkstores()
print(f"Darkstores: {len(darkstore_rows)}")

Darkstores: 100


### **Combine all Nodes and save**

In [7]:
all_nodes = mother_hub_rows + regional_hubs_rows + darkstore_rows
df_node = pd.DataFrame(all_nodes)
print(f"Total nodes: {len(df_node)}")
print(df_node["node_type"].value_counts())
print(df_node[["node_id", "node_type", "city", "tier", "parent_node_id"]].head(20))

Total nodes: 120
node_type
darkstore       100
regional_hub     12
mother_hub        8
Name: count, dtype: int64
            node_id     node_type       city          tier parent_node_id
0         MH_MUM_01    mother_hub     Mumbai  metropolitan           None
1         MH_DEL_01    mother_hub      Delhi  metropolitan           None
2         MH_BLR_01    mother_hub  Bangalore  metropolitan           None
3         MH_HYD_01    mother_hub  Hyderabad  metropolitan           None
4         MH_CHN_01    mother_hub    Chennai  metropolitan           None
5         MH_KOL_01    mother_hub    Kolkata  metropolitan           None
6         MH_PUN_01    mother_hub       Pune  metropolitan           None
7         MH_AMD_01    mother_hub  Ahmedabad  metropolitan           None
8    RH_MUM_WEST_01  regional_hub     Mumbai  metropolitan      MH_MUM_01
9    RH_MUM_EAST_01  regional_hub     Mumbai  metropolitan      MH_MUM_01
10  RH_DEL_NORTH_01  regional_hub      Delhi  metropolitan      MH_DEL_01

### **Generate dim_scanner**


In [8]:
sc_cfg = cfg["scanner"]
sz_cfg = cfg["scanner_zone_types"]
versions = sc_cfg["firmware_versions"]
rel_dates = sc_cfg["firmware_release_dates"]

def assign_firmware():
    """Older firmware is more likely"""
    weights = [0.30, 0.30, 0.25, 0.15] # v1.1.0 is most common
    return random.choices(versions, weights = weights, k=1)[0]

def assign_latency(is_degraded):
    if is_degraded:
        r = sc_cfg["latency_ms"]["degraded"]
    else:
        r = sc_cfg["latency_ms"]["good"]
    return random.randint(r[0], r[1])

def generate_scanners(df_node):
    scanners = []
    scan_num =1

    for _, node in df_node.iterrows():
        node_type =node["node_type"]
        tier = node["tier"]
        node_id = node["node_id"]

        # no. of scanners depending upon the node
        if node_type == "mother_hub":
            n_scanners = cfg["mother_hubs"]["scanners_per_node"]
        elif node_type == "regional_hub":
            n_scanners = cfg["regional_hubs"]["scanners_per_node"].get(tier,4)
        else:
            n_scanners = cfg["darkstores"]["scanners_per_node"].get(tier,1)
        
        ## assign scanner zones by fraction
        zone_types = list(sz_cfg.keys())
        fractions = [sz_cfg[z]["fraction"] for z in zone_types]
        counts = [max(1, round(f*n_scanners)) for f in fractions]

        # adjust for exactly n_scanners
        while sum(counts) > n_scanners:
            counts[counts.index(max(counts))] -=1
        while sum(counts) < n_scanners:
            counts[counts.index(min(counts))] +=1

        scanner_zone_list = []
        for zone, count in zip(zone_types, counts):
            scanner_zone_list.extend([zone]*count)

        for s_idx, scanner_zone in enumerate(scanner_zone_list, 1):
            firmware = assign_firmware()
            rel_date = date.fromisoformat(rel_dates[firmware])
            is_degraded = random.random() < sc_cfg["degraded_fraction"]
            has_rtc = random.random() > sc_cfg["dead_rtc_fraction"]
            latency = assign_latency(is_degraded)

            # calibrated 1-18 months before the simulation start
            days_since_cal = random.randint(30,540)
            last_cal = SIM_START - timedelta(days=days_since_cal)
            failures = 0
            if is_degraded:
                failures = random.randint(1,8)

            # node zone code for device_id
            node_code = node_id.replace("DS_", "").replace("RH_", "") \
            .replace("MH_", "").replace("_01","").replace("_02","")

            raw = {
                "device_id": f"SCAN_{node_code}_{s_idx:03d}",
                "assigned_node_id": node_id,
                "scanner_zone": scanner_zone,
                "firmware_version": firmware,
                "firmware_release_date": rel_date,
                "battery_backed_rtc": has_rtc,
                "avg_sync_latency_ms": latency,
                "is_degraded": is_degraded,
                "chaos_affinity": sz_cfg[scanner_zone]["chaos_affinity"],
                "last_calibrated": last_cal,
                "failure_count_30d": failures,
            }

            validated = ScannerSchema(**raw)
            scanners.append(validated.model_dump())
            scan_num +=1
        
    return pd.DataFrame(scanners)
df_scanner = generate_scanners(df_node)
print(f"Scanners: {len(df_scanner)}")
print(df_scanner["scanner_zone"].value_counts())



Scanners: 424
scanner_zone
stock_count    128
returns        128
outbound       114
inbound         54
Name: count, dtype: int64


### **Generate dim_hub_hierarchy**

In [9]:
def generate_hub_hierarchy(df_node):
    rows = []
    node_lookup = df_node.set_index("node_id").to_dict("index")
    darkstores = df_node[df_node["node_type"] == "darkstore"]

    for _, ds in darkstores.iterrows():
        rh_id = ds["parent_node_id"]
        if rh_id not in node_lookup:
            print(f"WARNING: {ds['node_id']} has missing parent {rh_id}")
            continue

        rh = node_lookup[rh_id]
        mh_id = rh["parent_node_id"]

        if mh_id not in node_lookup:
            print(f"WARNING: {rh_id} has missing parent {mh_id}")
            continue
        mh = node_lookup[mh_id]

        raw = {
            "darkstore_id": ds["node_id"],
            "darkstore_zone": ds["zone"] or "",
            "darkstore_city": ds["city"],
            "tier": ds["tier"],
            "regional_hub_id": rh_id,
            "regional_hub_city": rh["city"],
            "mother_hub_id": mh_id,
            "mother_hub_zone": mh["zone"] or ""
        }
        validated = HubHierarchySchema(**raw)
        rows.append(validated.model_dump())


    return pd.DataFrame(rows)
df_hierarchy = generate_hub_hierarchy(df_node)
print(f"Hierarchy rows: {len(df_hierarchy)}")
print(df_hierarchy.head())

Hierarchy rows: 100
  darkstore_id darkstore_zone darkstore_city          tier regional_hub_id  \
0    DS_AND_01        Andheri         Mumbai  metropolitan  RH_MUM_WEST_01   
1    DS_BAN_01         Bandra         Mumbai  metropolitan  RH_MUM_WEST_01   
2    DS_TNA_01          Thane         Mumbai  metropolitan  RH_MUM_EAST_01   
3    DS_KUR_01          Kurla         Mumbai  metropolitan  RH_MUM_EAST_01   
4    DS_MLD_01          Malad         Mumbai  metropolitan  RH_MUM_WEST_01   

  regional_hub_city mother_hub_id mother_hub_zone  
0            Mumbai     MH_MUM_01        Bhiwandi  
1            Mumbai     MH_MUM_01        Bhiwandi  
2            Mumbai     MH_MUM_01        Bhiwandi  
3            Mumbai     MH_MUM_01        Bhiwandi  
4            Mumbai     MH_MUM_01        Bhiwandi  


### **Save All Tables**

In [10]:
import os
os.makedirs("data", exist_ok=True)

df_supplier.to_csv("D:/OMNINODE_RESILIENCE_PIPELINE/Data_generation/Production_data/dim_supplier.csv", index=False)
df_product.to_csv("D:/OMNINODE_RESILIENCE_PIPELINE/Data_generation/Production_data/dim_product.csv", index=False)
df_node.to_csv("D:/OMNINODE_RESILIENCE_PIPELINE/Data_generation/Production_data/dim_node.csv", index=False)
df_scanner.to_csv("D:/OMNINODE_RESILIENCE_PIPELINE/Data_generation/Production_data/dim_scanner.csv", index=False)
df_hierarchy.to_csv("D:/OMNINODE_RESILIENCE_PIPELINE/Data_generation/Production_data/dim_hub_hierarchy.csv", index=False)
print("All tables saved.")

print(f"dim_supplier: {len(df_supplier):>5} rows")
print(f"dim_node: {len(df_node):>5} rows")
print(f"dim_scanner: {len(df_scanner):>5} rows")
print(f"dim_hub_hierarchy: {len(df_hierarchy):>5} rows")


All tables saved.
dim_supplier:    19 rows
dim_node:   120 rows
dim_scanner:   424 rows
dim_hub_hierarchy:   100 rows
